# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors including MSI-H Status and Anatomical Distribution Exploration with `mlcroissant`

This notebook provides a template for loading and exploring the FAIR^2 dataset using the `mlcroissant` library, referencing all entities by their `@id` fields for reproducible and standardized analysis.

### Dataset Source

The dataset source is provided via Croissant schema URL:

`https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json`


In [ ]:
# Ensure mlcroissant is installed
!pip install mlcroissant

## 1. Data Loading

Load metadata and records from the dataset using the `mlcroissant` library.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the dataset Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json'

# Load the dataset using mlcroissant
dataset = mlc.Dataset(croissant_url)

# Access dataset metadata
metadata = dataset.metadata
print(f"Dataset Name: {metadata.name}")
print(f"Dataset Description: {metadata.description}\n")
print(f"Dataset Identifier (@id): {metadata['@id']}")
print(f"Dataset License: {metadata.license}")
print(f"Dataset Version: {metadata.version}")
print(f"Dataset Temporal Coverage: {metadata.temporalCoverage}")
print(f"Dataset Keywords: {metadata.keywords}")

## 2. Data Overview

Review available record sets, fields, and their IDs.
Entities (record sets, fields, columns) are referenced strictly by their `@id` values.

In [ ]:
# List all record sets with their @id
record_sets = dataset.record_sets()

print("Available Record Sets:")
for rs in record_sets:
    print(f"  - {rs['@id']} | {rs.get('name', '[no name]')} | {rs.get('description', '[no description]')}")

# Pick a record set @id for illustration
if len(record_sets):
    example_record_set_id = record_sets[0]['@id']
else:
    example_record_set_id = None

# List fields/columns from the record set using their @id
if example_record_set_id:
    print(f"\nFields/Columns for Record Set '@id': {example_record_set_id}")
    fields = dataset.fields(record_set=example_record_set_id)
    for fld in fields:
        print(f"  - {fld['@id']} | {fld.get('name', '[no name]')} | {fld.get('description', '[no description]')}")

In [ ]:
# Print first few records for selected record set using its @id
if example_record_set_id:
    print(f"\nFirst 3 records in record set {example_record_set_id}:")
    for i, record in enumerate(dataset.records(record_set=example_record_set_id)):
        if i >= 3:
            break
        print(f"Record {i+1}: {record}")

## 3. Data Extraction

Load data from each record set into a pandas DataFrame for analysis.

Use the record set and field `@id`s obtained from above overview. All variable references follow the `@id` convention.

In [ ]:
# Extract data from each record set
dataframes = {}

# Collect all available record set @id values
record_set_ids = [rs['@id'] for rs in record_sets]
print(f"Record Set IDs: {record_set_ids}")

for record_set_id in record_set_ids:
    records = list(dataset.records(record_set=record_set_id))
    if records:  # If records available
        df = pd.DataFrame(records)
        dataframes[record_set_id] = df
        print(f"\nColumns in Record Set {record_set_id}: {df.columns.tolist()}")
        print(df.head())
    else:
        print(f"\nRecord Set {record_set_id} has no records.")

## 4. Exploratory Data Analysis (EDA)

Apply common data processing steps: filtering, normalization, grouping. All fields are referenced explicitly by their `@id`.

*If you have multiple record sets or numeric fields, you may adjust the IDs and logic accordingly.*

In [ ]:
# Choose a record set and a numeric field for EDA

# Find first record set with at least one numeric field
sel_record_set_id = None
numeric_field_id = None
group_field_id = None

for record_set_id, df in dataframes.items():
    # Check dataframe column types
    for col in df.columns:
        if pd.api.types.is_numeric_dtype(df[col]):
            sel_record_set_id = record_set_id
            numeric_field_id = col
            break
    if sel_record_set_id:
        # Use first non-numeric column for grouping if possible
        non_numeric_cols = [c for c in df.columns if not pd.api.types.is_numeric_dtype(df[c])]
        if non_numeric_cols:
            group_field_id = non_numeric_cols[0]
        break

if sel_record_set_id and numeric_field_id:
    print(f"Using Record Set @id: {sel_record_set_id}")
    print(f"Numeric Field @id: {numeric_field_id}")
    if group_field_id:
        print(f"Grouping Field @id: {group_field_id}")

    df = dataframes[sel_record_set_id]
    threshold = 10

    # Filter records based on numeric field threshold
    filtered_df = df[df[numeric_field_id] > threshold]
    print(f"\nFiltered records with {numeric_field_id} > {threshold}:")
    print(filtered_df.head())

    # Normalize numeric field
    norm_col = f"{numeric_field_id}_normalized"
    filtered_df[norm_col] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
    print(f"\nNormalized {numeric_field_id} for filtered records:")
    print(filtered_df[[numeric_field_id, norm_col]].head())

    # Group by chosen field and show stats
    if group_field_id and group_field_id in filtered_df.columns:
        grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
        print(f"\nGrouped data by {group_field_id} (mean {numeric_field_id}):")
        print(grouped_df.head())
else:
    print("No numeric fields found in any record set. Please check data availability.")

## 5. Visualization

Visualize data distributions or relationships between fields in the dataset.

*Field and record set references are by `@id` only.*

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# If we have an example field and filtered dataframe, visualize
if sel_record_set_id and numeric_field_id:
    filtered_df = dataframes[sel_record_set_id][dataframes[sel_record_set_id][numeric_field_id] > 10]
    plt.figure(figsize=(8,5))
    sns.histplot(filtered_df[numeric_field_id], bins=10, kde=True)
    plt.title(f'Distribution of field {numeric_field_id} (>10)')
    plt.xlabel(numeric_field_id)
    plt.ylabel('Count')
    plt.show()

    if group_field_id and group_field_id in filtered_df.columns:
        plt.figure(figsize=(8,5))
        sns.boxplot(x=group_field_id, y=numeric_field_id, data=filtered_df)
        plt.title(f'{numeric_field_id} by {group_field_id}')
        plt.xlabel(group_field_id)
        plt.ylabel(numeric_field_id)
        plt.show()
else:
    print("No numeric fields for visualization.")

## 6. Conclusion

- The FAIR^2 dataset is loaded and explored using `mlcroissant`, referencing all entities by their standard `@id`.
- Tabular data from each record set is extracted and processed in pandas DataFrames, allowing flexible data analysis and EDA.
- Basic filtering, normalization, grouping, and visualization are demonstrated, adaptable to any Croissant-compliant dataset.
- All operations are reproducible and FAIR by referencing data elements consistently via schema IDs.